FEATURE SELECTION

AIM: To perform feature selection on the labeled FRA Atlas dataset and identify the most relevant features for predicting `pmksy_score`.   

1) Imports and Loading Dataset 

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv("../ml/data/labeled_dataset.csv")

df.head()

,plot_id,state,district,lat,lon,land_area_ha,claim_type,soil_ph,soil_organic_carbon,soil_clay_pct,...,soil_ph_imputed,soil_organic_carbon_imputed,soil_clay_pct_imputed,elevation_m_imputed,pm_kisan_score,mgnrega_score,jjm_score,pmfby_score,pmksy_score,soil_health_card_score
0,SYN-00001,Madhya Pradesh,Chhindwara,22.03435,78.92785,2.63,IFR,7.1,17.22,43.96,...,True,True,True,False,84.28,53.77,63.18,59.21,60.99,11.51
1,SYN-00002,Madhya Pradesh,Chhindwara,22.04054,78.91853,1.25,IFR,7.4,14.40,46.00,...,False,False,False,False,86.66,53.30,62.54,55.70,67.43,22.15
2,SYN-00003,Madhya Pradesh,Chhindwara,22.06489,78.91472,3.00,CR,7.4,14.40,45.10,...,False,False,False,True,25.28,50.69,59.42,48.91,64.49,22.74
3,SYN-00004,Madhya Pradesh,Chhindwara,22.04481,78.94029,3.59,IFR,7.1,17.22,43.96,...,True,True,True,False,85.21,54.10,67.04,46.28,60.25,19.46
4,SYN-00005,Madhya Pradesh,Chhindwara,22.06331,78.95167,1.80,IFR,7.1,17.22,43.96,...,True,True,True,True,89.37,53.43,66.11,56.49,69.63,13.05


2) Define X and y: The selected input features are used as X, while `pmksy_score` is used as the target variable y.

In [2]:
features = [
    "soil_ph",
    "soil_organic_carbon",
    "soil_clay_pct",
    "elevation_m",
    "groundwater_stress_pct",
    "rainfall_mm_avg",
    "land_area_ha",
    "is_scheduled_area"
]

X = df[features]
y = df["pmksy_score"]

X.head()

,soil_ph,soil_organic_carbon,soil_clay_pct,elevation_m,groundwater_stress_pct,rainfall_mm_avg,land_area_ha,is_scheduled_area
0,7.1,17.22,43.96,667.0,58.0,1150.0,2.63,True
1,7.4,14.40,46.00,675.0,58.0,1150.0,1.25,True
2,7.4,14.40,45.10,660.8,58.0,1150.0,3.00,True
3,7.1,17.22,43.96,666.0,58.0,1150.0,3.59,True
4,7.1,17.22,43.96,660.8,58.0,1150.0,1.80,True


3) Correlation Analysis: Correlation measures the linear relationship between each feature and
the target variable `pmksy_score`. A higher absolute correlation indicates a stronger linear relationship.

In [3]:
correlation = df[features + ["pmksy_score"]].corr()["pmksy_score"]

correlation.sort_values(ascending=False)

pmksy_score               1.000000
groundwater_stress_pct    0.856626
soil_ph                   0.849027
soil_clay_pct             0.348608
elevation_m              -0.156451
land_area_ha             -0.164830
soil_organic_carbon      -0.655179
rainfall_mm_avg          -0.765649
is_scheduled_area              NaN
Name: pmksy_score, dtype: float64

4) Univariate Selection - SelectKBest:
SelectKBest evaluates each feature individually using the `f_regression`
statistical test and selects the features with the highest scores.

In [4]:
selector = SelectKBest(score_func=f_regression, k="all")

selector.fit(X, y)

univariate_results = pd.DataFrame({
    "Feature": X.columns,
    "F-Score": selector.scores_
})

univariate_results.sort_values(
    by="F-Score",
    ascending=False
)

,Feature,F-Score
4,groundwater_stress_pct,4239.801945
0,soil_ph,3971.525918
5,rainfall_mm_avg,2178.930479
1,soil_organic_carbon,1156.744939
2,soil_clay_pct,212.766156
6,land_area_ha,42.952748
3,elevation_m,38.589898
7,is_scheduled_area,0.000000


5) Wrapper Method (RFE + LinearRegression): Recursive Feature Elimination (RFE) repeatedly removes less important features using a Linear Regression model. The features with lower rank values are considered more important.

In [5]:
model = LinearRegression()

rfe = RFE(
    estimator=model,
    n_features_to_select=4
)

rfe.fit(X, y)

rfe_results = pd.DataFrame({
    "Feature": X.columns,
    "RFE_Ranking": rfe.ranking_,
    "Selected": rfe.support_
})

rfe_results.sort_values(by="RFE_Ranking")

,Feature,RFE_Ranking,Selected
0,soil_ph,1,True
2,soil_clay_pct,1,True
6,land_area_ha,1,True
4,groundwater_stress_pct,1,True
1,soil_organic_carbon,2,False
5,rainfall_mm_avg,3,False
3,elevation_m,4,False
7,is_scheduled_area,5,False


7) Comparison Table Across All Methods: The results from correlation analysis, SelectKBest, RFE, and Decision Tree feature importance are combined for comparison.

In [7]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)

tree.fit(X, y)

tree_results = pd.DataFrame({
    "Feature": X.columns,
    "Tree Importance": tree.feature_importances_
})

tree_results.sort_values(
    by="Tree Importance",
    ascending=False
)

,Feature,Tree Importance
4,groundwater_stress_pct,0.814962
6,land_area_ha,0.074107
5,rainfall_mm_avg,0.055600
2,soil_clay_pct,0.017164
1,soil_organic_carbon,0.016704
3,elevation_m,0.015355
0,soil_ph,0.006108
7,is_scheduled_area,0.000000


In [8]:
comparison = pd.DataFrame({
    "Feature": X.columns,
    "Correlation": [
        correlation[feature] for feature in X.columns
    ],
    "F-Score": selector.scores_,
    "RFE Ranking": rfe.ranking_,
    "Decision Tree Importance": tree.feature_importances_
})

comparison

,Feature,Correlation,F-Score,RFE Ranking,Decision Tree Importance
0,soil_ph,0.849027,3971.525918,1,0.006108
1,soil_organic_carbon,-0.655179,1156.744939,2,0.016704
2,soil_clay_pct,0.348608,212.766156,1,0.017164
3,elevation_m,-0.156451,38.589898,4,0.015355
4,groundwater_stress_pct,0.856626,4239.801945,1,0.814962
5,rainfall_mm_avg,-0.765649,2178.930479,3,0.055600
6,land_area_ha,-0.164830,42.952748,1,0.074107
7,is_scheduled_area,NaN,0.000000,5,0.000000


## Conclusion

Feature selection was performed using four different methods:
Correlation Analysis, SelectKBest, RFE with Linear Regression, and
Decision Tree feature importance.

The methods provide different measures of feature relevance and can be
compared to identify features that are useful for predicting `pmksy_score`.